In [5]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from google import genai #type: ignore
from google.genai import types #type: ignore
from google.colab import drive #type: ignore
from dotenv import load_dotenv
import os

drive.mount("/content/drive")
load_dotenv("/content/drive/MyDrive/.env")
api_key = os.getenv("GOOGLE_API_KEY")  


Mounted at /content/drive


In [6]:
client = genai.Client(api_key=api_key)

MODEL_ID = "gemini-2.5-flash"

In [ ]:
document_1 = {"title": "Porsche un mythe ?",
            "content": """Dr. Ing. hc F. Porsche AG[a], généralement abrégé en Porsche (prononcé en allemand :est un constructeur d'automobiles de sport allemand.
La société fut fondée en 1931 par Ferdinand Porsche, puis reprise par son fils Ferry Porsche. Ferdinand Porsche est l'ingénieur qui créa la première Volkswagen. 
Les principales usines du constructeur sont situées à Leipzig et à Zuffenhausen, en Allemagne.
C'est chronologiquement la dixième marque à avoir intégré le groupe Volkswagen AG en 2009. En 2023, ses ventes mondiales se sont élevées à plus de 320 000 véhicules, en hausse par rapport à 2022."""
            }

document_2 = { "title": "Ferrari, l'histoire d'une légende",
              "content": """Ferrari S.p.A. est un constructeur automobile italien installé à Maranello en Italie, fondée par Enzo Ferrari en 1947.
L'histoire de Ferrari est indissociable de celle de la Scuderia Ferrari, écurie automobile évoluant en Sport-prototypes tout comme en grand tourisme – et plus tard en Formule 1
depuis 1929, avec laquelle le constructeur a connu ses plus grands succès. 
Forte de son expérience en compétition, la marque au « cheval cabré » (« cavallino rampante ») y puise les techniques équipant ses modèles de série, 
comme l'attestent les Ferrari 288 GTO, F50 ou encore Ferrari Enzo, aux performances sportives exceptionnelles lors de leur lancement."""
}

document_3 = { "title": "Mercedes, la marque des patrons",
              "content": """Mercedes-Benz est une marque allemande d'automobiles (modèles premium, de sport et de luxe), de camions, d'autocars et d'autobus indépendante, fondée en 1926 par trois constructeurs : Daimler-Motoren-Gesellschaft, Mercedes et Benz & Cie. 
              Mercedes-Benz est également établie depuis 2019 comme constructeur automobile à part entière sous la dénomination Mercedes-Benz AG, filiale de Mercedes-Benz Group, depuis la décision de Daimler AG de se scinder en deux entités distinctes.
L'entreprise Daimler a établi sa réputation en sport automobile, dans les années 1930, puis dans le championnat du monde de Formule 1 jusqu'en 1955, et à nouveau depuis 2010, où elle domine l'ère des moteurs turbo-hybrides avec huit sacres consécutifs chez les constructeurs de 2014 à 2021."""
}


documents = [document_1, document_2, document_3]

combined_content = "\n\n".join([f"Source Title: {document['title']}\nContent: {document['content']}" for document in documents])


Source Title: Porsche un mythe ?
Content: Dr. Ing. hc F. Porsche AG[a], généralement abrégé en Porsche (prononcé en allemand :est un constructeur d'automobiles de sport allemand.
La société fut fondée en 1931 par Ferdinand Porsche, puis reprise par son fils Ferry Porsche. Ferdinand Porsche est l'ingénieur qui créa la première Volkswagen. 
Les principales usines du constructeur sont situées à Leipzig et à Zuffenhausen, en Allemagne.
C'est chronologiquement la dixième marque à avoir intégré le groupe Volkswagen AG en 2009. En 2023, ses ventes mondiales se sont élevées à plus de 320 000 véhicules, en hausse par rapport à 2022.

Source Title: Ferrari, l'histoire d'une légende
Content: Ferrari S.p.A. est un constructeur automobile italien installé à Maranello en Italie, fondée par Enzo Ferrari en 1947.
L'histoire de Ferrari est indissociable de celle de la Scuderia Ferrari, écurie automobile évoluant en Sport-prototypes tout comme en grand tourisme – et plus tard en Formule 1
depuis 1929, a

str

In [7]:
class FormatQuestionList(BaseModel):
    question: list[str] = Field(description="A list of questions regarding something in the text")
    metadata: list[str] = Field(description="Keywords related to the question")


In [10]:
PROMPT_QUESTION= """Your task is to generate {nb_question} questions regarding the document provided below.
    
    
    <document>
    {content}
    </document>
    
    """.strip()


In [ ]:
def generate_questions(content: str, nb_question: int):
    
    
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=FormatQuestionList)
    
    questions = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT_QUESTION.format(content=content, nb_question=nb_question),
    config=config
    )

    return questions.parsed.question

    
question = generate_questions(combined_content, 5)


In [18]:
PROMPT_RESPONSE = """Provide the answer to the following question based only on the document provided below.{question}


<document>
{document}
</document>
""".strip()

In [19]:
def generate_answer(document: str, question: str ):
    
    
    question_answer = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT_RESPONSE.format(document=document, question=question),
    )
    
    return question_answer.text


answer = generate_answer(combined_content,question[0])

In [22]:
PROMPT_SOURCE = """Provide the source to this response {response} based only on the document provided below.


<document>
{document}
</document>
""".strip()

In [25]:
def find_source(document: str, response: str ):
    
    
    question_source = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT_SOURCE.format(document=document, response=response),
    )
    
    return question_source.text


source = find_source(combined_content,answer)

In [28]:
import time

def sequential_workflow(content: str, nb_questions: int):
    
    results = []
    questions = generate_questions(content=content, nb_question=nb_questions)
    
    for question in questions:
        answer = generate_answer(content, question)
        sources = find_source(content, question)
        
        results.append({
            "question": question,
            "answer": answer,
            "sources": sources
        })
    
    return results

start = time.monotonic()
results = sequential_workflow(combined_content, 5)
end = time.monotonic()
print(f"Temps: {end - start:.2f} secondes")
    

Temps: 20.39 secondes
